# YOLO для видео: подготовка данных, дообучение и детектирование

Этот ноутбук сделан как набор блоков, которые можно забирать по отдельности:
- подготовка путей;
- генерация `dataset.yaml`;
- запуск дообучения;
- инференс на видео;
- покадровая обработка и сохранение ролика.

Шаблон ориентирован на лёгкую модель YOLO и быстрый старт.

## Что покрывает этот ноутбук

1. Структура датасета для детекции  
2. Проверка путей и генерация `dataset.yaml`  
3. Дообучение модели на своих данных  
4. Быстрый инференс на готовом видео  
5. Покадровая обработка видео через OpenCV  
6. Сохранение результата в новый видеофайл

In [ ]:
# =========================
# БЛОК 1. Импорты
# =========================

from pathlib import Path
import yaml

import cv2
import matplotlib.pyplot as plt
from ultralytics import YOLO

In [ ]:
# =========================
# БЛОК 2. Основные настройки
# Это главный блок для адаптации под свои данные.
# =========================

# Базовые веса. Для быстрого старта обычно удобно начать с nano-версии.
BASE_WEIGHTS = "yolov8n.pt"

# Корень датасета в формате YOLO.
# Ожидается структура:
# dataset/
#   images/
#       train/
#       val/
#   labels/
#       train/
#       val/
DATASET_ROOT = Path("data/yolo_dataset")

# Имена классов.
# Пример: ["person", "helmet", "vest"]
CLASS_NAMES = ["object"]

# Путь к готовому видео для детекции.
VIDEO_PATH = Path("data/videos/example.mp4")

# Куда сохранять результаты.
OUTPUT_DIR = Path("outputs/video_pipeline")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Параметры обучения.
EPOCHS = 10
IMG_SIZE = 640
BATCH_SIZE = 8
DEVICE = "cpu"   # если есть GPU, можно указать '0'

# Параметры инференса на видео.
CONF_THRESHOLD = 0.25

print("DATASET_ROOT =", DATASET_ROOT)
print("VIDEO_PATH   =", VIDEO_PATH)
print("OUTPUT_DIR   =", OUTPUT_DIR)
print("CLASS_NAMES  =", CLASS_NAMES)

## Формат датасета

Для детекции YOLO обычно используется такая структура:

```text
data/yolo_dataset/
├── images/
│   ├── train/
│   └── val/
└── labels/
    ├── train/
    └── val/
```

Для каждого изображения должен быть `.txt` файл с разметкой в формате:

```text
class_id x_center y_center width height
```

Все координаты нормированы в диапазон от 0 до 1.

In [ ]:
# =========================
# БЛОК 3. Проверка структуры датасета
# Этот блок полезно запускать первым перед обучением.
# =========================

def check_dataset_structure(dataset_root):
    dataset_root = Path(dataset_root)

    required_dirs = [
        dataset_root / "images" / "train",
        dataset_root / "images" / "val",
        dataset_root / "labels" / "train",
        dataset_root / "labels" / "val",
    ]

    for directory in required_dirs:
        print(f"{directory}: {'OK' if directory.exists() else 'MISSING'}")

check_dataset_structure(DATASET_ROOT)

In [ ]:
# =========================
# БЛОК 4. Генерация dataset.yaml
# Этот файл нужен YOLO для обучения.
# =========================

def create_dataset_yaml(dataset_root, class_names, output_path=None):
    dataset_root = Path(dataset_root)

    data_config = {
        "path": str(dataset_root.resolve()),
        "train": "images/train",
        "val": "images/val",
        "names": {idx: name for idx, name in enumerate(class_names)},
    }

    if output_path is None:
        output_path = dataset_root / "dataset.yaml"

    with open(output_path, "w", encoding="utf-8") as f:
        yaml.safe_dump(data_config, f, allow_unicode=True, sort_keys=False)

    return Path(output_path)

dataset_yaml_path = create_dataset_yaml(DATASET_ROOT, CLASS_NAMES)
print("dataset.yaml создан:", dataset_yaml_path)
print(dataset_yaml_path.read_text(encoding="utf-8"))

In [ ]:
# =========================
# БЛОК 5. Загрузка модели
# При переносе в другой проект обычно меняется только путь к весам.
# =========================

model = YOLO(BASE_WEIGHTS)
model

## Дообучение на своих данных

Этот блок запускает обучение на предоставленном датасете.  
Его удобно использовать как стартовую основу, а дальше уже подстраивать:
- `epochs`
- `imgsz`
- `batch`
- `device`

Если данных мало, обычно имеет смысл начать с короткого прогона, чтобы быстро проверить пайплайн.

In [ ]:
# =========================
# БЛОК 6. Обучение модели
# Здесь минимальный и понятный запуск train().
# =========================

# Раскомментируйте блок ниже, когда датасет уже готов.
# train_results = model.train(
#     data=str(dataset_yaml_path),
#     epochs=EPOCHS,
#     imgsz=IMG_SIZE,
#     batch=BATCH_SIZE,
#     device=DEVICE,
#     project=str(OUTPUT_DIR),
#     name="train_run",
#     exist_ok=True
# )

print("После подготовки датасета снимите комментарии и запустите обучение.")

## Быстрый инференс на видео одной командой

Это самый короткий путь:
- указать путь к видео;
- получить новый ролик с отрисованными боксами;
- сохранить результат в папку.

Такой блок очень удобно переносить в другой проект почти без изменений.

In [ ]:
# =========================
# БЛОК 7. Быстрый инференс на видео через YOLO
# =========================

# Можно использовать либо исходные веса, либо веса после дообучения.
# Например:
# trained_weights = OUTPUT_DIR / "train_run" / "weights" / "best.pt"
# video_model = YOLO(str(trained_weights))

video_model = model

# Раскомментируйте после появления реального видео:
# video_results = video_model.predict(
#     source=str(VIDEO_PATH),
#     conf=CONF_THRESHOLD,
#     imgsz=IMG_SIZE,
#     device=DEVICE,
#     save=True,
#     project=str(OUTPUT_DIR),
#     name="video_predict",
#     exist_ok=True
# )

print("Подставьте реальный VIDEO_PATH и снимите комментарии для быстрого видео-инференса.")

## Покадровая обработка видео

Ниже более явный и понятный вариант.  
Он полезен, если нужно:
- вставить свою бизнес-логику между кадрами;
- фильтровать классы;
- считать объекты;
- писать поверх кадра свой текст;
- сохранять промежуточные данные.

In [ ]:
# =========================
# БЛОК 8. Вспомогательные функции для работы с видео
# =========================

def open_video_capture(video_path):
    video_path = Path(video_path)
    if not video_path.exists():
        raise FileNotFoundError(f"Видео не найдено: {video_path}")

    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise RuntimeError(f"Не удалось открыть видео: {video_path}")
    return cap


def get_video_writer(output_path, fps, frame_width, frame_height):
    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    writer = cv2.VideoWriter(str(output_path), fourcc, fps, (frame_width, frame_height))
    return writer


def process_video_frame_by_frame(model, video_path, output_path, conf=0.25, imgsz=640, device="cpu", max_frames=None):
    """Покадровая обработка видео и сохранение нового ролика.

    max_frames полезен для быстрой отладки:
    - можно прогнать только первые 50 или 100 кадров;
    - проверить, что пайплайн работает;
    - затем убрать ограничение.
    """
    cap = open_video_capture(video_path)

    fps = cap.get(cv2.CAP_PROP_FPS)
    frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    writer = get_video_writer(output_path, fps, frame_width, frame_height)

    frame_count = 0

    try:
        while True:
            ret, frame = cap.read()
            if not ret:
                break

            results = model.predict(
                source=frame,
                conf=conf,
                imgsz=imgsz,
                device=device,
                verbose=False
            )

            plotted_frame = results[0].plot()
            writer.write(plotted_frame)

            frame_count += 1
            if max_frames is not None and frame_count >= max_frames:
                break

        print(f"Обработано кадров: {frame_count}")
        print(f"Результат сохранён: {output_path}")

    finally:
        cap.release()
        writer.release()

In [ ]:
# =========================
# БЛОК 9. Запуск покадровой обработки
# =========================

output_video_path = OUTPUT_DIR / "processed_video.mp4"

# Раскомментируйте, когда будет реальное видео:
# process_video_frame_by_frame(
#     model=video_model,
#     video_path=VIDEO_PATH,
#     output_path=output_video_path,
#     conf=CONF_THRESHOLD,
#     imgsz=IMG_SIZE,
#     device=DEVICE,
#     max_frames=100  # для быстрой проверки можно ограничить число кадров
# )

print("После подстановки видео можно получить новый ролик с детекциями.")

In [ ]:
# =========================
# БЛОК 10. Просмотр первого кадра из видео
# Этот блок помогает быстро проверить входной файл.
# =========================

def show_first_video_frame(video_path, figsize=(10, 6)):
    cap = open_video_capture(video_path)
    ret, frame = cap.read()
    cap.release()

    if not ret:
        raise RuntimeError("Не удалось прочитать первый кадр.")

    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    plt.figure(figsize=figsize)
    plt.imshow(frame_rgb)
    plt.title("First video frame")
    plt.axis("off")
    plt.show()

# show_first_video_frame(VIDEO_PATH)
print("Снимите комментарии, чтобы быстро посмотреть первый кадр видео.")

## Что менять при переносе в другой проект

Обычно достаточно поменять:
- `BASE_WEIGHTS`
- `DATASET_ROOT`
- `CLASS_NAMES`
- `VIDEO_PATH`
- `EPOCHS`, `IMG_SIZE`, `BATCH_SIZE`
- `CONF_THRESHOLD`

Остальные блоки можно использовать почти без изменений.

## Практический совет

Сначала удобно пройти такой путь:
1. проверить структуру датасета;
2. создать `dataset.yaml`;
3. сделать очень короткое обучение;
4. проверить инференс на одном видео;
5. только потом усложнять логику.

Так проще быстрее поймать ошибки в путях, разметке и настройках.